<div style="font-size:22pt; line-height:25pt; font-weight:bold; text-align:center;">Notebook 2 — Dynamic Programming and Value-Based RL</div>

<div class="alert alert-success">

**Learning outcomes:**
By the end of this notebook you should be able to:
- state and apply the Bellman evaluation and optimality equations,
- implement value iteration for tabular MDPs,
- explain approximate value iteration and its connection to Q-learning,
- implement a Deep Q-Network (DQN) with experience replay and target networks,
- describe the key ideas of Soft Actor-Critic (SAC) for continuous action spaces.
</div>

# Bellman Equations

## Bellman evaluation equation

A key property of $v^\pi$ (and $q^\pi$) is that they satisfy a fixed-point equation.

<div class="alert alert-success">

**Bellman evaluation equation for $q^\pi$:**
$$q^\pi(s,a) = r(s,a) + \gamma \sum_{s'} p(s'|s,a) \sum_{a'} \pi(a'|s')\, q^\pi(s',a')$$

Or in operator notation: $q^\pi = \mathbb{T}^\pi q^\pi$, where $\mathbb{T}^\pi$ is the **Bellman evaluation operator**.
</div>

Intuition: the value of $(s,a)$ = immediate reward + discounted value of where we end up.

## Bellman optimality equation

<div class="alert alert-success">

**Bellman optimality equation for $q^*$:**
$$q^*(s,a) = r(s,a) + \gamma \sum_{s'} p(s'|s,a) \max_{a'} q^*(s',a')$$

Or: $q^* = \mathbb{T}^* q^*$, where $\mathbb{T}^*$ is the **Bellman optimality operator**.
</div>

Both operators are **contractions** with modulus $\gamma < 1$, so iterating them from any starting point converges to the unique fixed point.

Overall: the repeated application of $\mathbb{T}^\pi$ / $\mathbb{T}^*$ is a **[Dynamic Programming](https://gwern.net/doc/statistics/decision/1957-bellman-dynamicprogramming.pdf)** procedure, that converges to $q^\pi$ / $q^*$.

# Value Iteration

**Value iteration** repeatedly applies $\mathbb{T}^*$:
$$q_{n+1} \leftarrow \mathbb{T}^* q_n, \quad q_0 = 0.$$

Convergence is guaranteed: $\|q_n - q^*\|_\infty \leq \gamma^n \|q_0 - q^*\|_\infty$.

<div class="alert alert-warning">
Write a pseudo-code for Value Iteration.
</div>

In [ ]:
import gymnasium as gym
import gymnasium.envs.toy_text.frozen_lake as fl
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

env = gym.make('FrozenLake-v1', render_mode="ansi")
env.reset()

n_states  = env.observation_space.n
n_actions = env.action_space.n
gamma = 0.99

In [ ]:
def value_iteration(env, gamma, tol=1e-6, max_iter=5000):
    """Value iteration. Returns Q* and convergence residuals."""
    n_s = env.observation_space.n
    n_a = env.action_space.n
    Q = np.zeros((n_s, n_a))
    residuals = []

    for _ in range(max_iter):
        Q_new = np.zeros_like(Q)
        for s in range(n_s):
            for a in range(n_a):
                for prob, s_next, reward, done in env.unwrapped.P[s][a]:
                    if done:
                        Q_new[s, a] += prob * reward
                    else:
                        Q_new[s, a] += prob * (reward + gamma * np.max(Q[s_next]))
        residual = np.max(np.abs(Q_new - Q))
        residuals.append(residual)
        Q = Q_new
        if residual < tol:
            break

    return Q, np.array(residuals)

Q_star, residuals = value_iteration(env, gamma)
print(f"Converged in {len(residuals)} iterations, final residual {residuals[-1]:.2e}")

In [ ]:
# Visualise convergence
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.plot(residuals)
plt.xlabel("Iteration")
plt.ylabel("Max Bellman residual")
plt.title("Value iteration convergence")

plt.subplot(1, 2, 2)
plt.semilogy(residuals)
plt.xlabel("Iteration")
plt.ylabel("Residual (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
# Extract optimal policy and show it
actions_symbol = {fl.LEFT: '←', fl.DOWN: '↓', fl.RIGHT: '→', fl.UP: '↑'}
pi_star = np.argmax(Q_star, axis=1)

print("Optimal Q* (reshaped 4×4 per action is hard to show; showing V* = max_a Q*):")
print(np.max(Q_star, axis=1).reshape(4,4).round(3))
print()
print("Optimal policy π*:")
for row in range(4):
    print("  " + "  ".join(actions_symbol[pi_star[row*4+col]] for col in range(4)))

In [ ]:
# Evaluate the policy found by value iteration
def run_policy(env, policy, n=10_000, horizon=200, gamma=0.99):
    returns = []
    for _ in range(n):
        s, _ = env.reset()
        G, disc = 0.0, 1.0
        for _ in range(horizon):
            a = int(policy[s])
            s, r, done, trunc, _ = env.step(a)
            G += disc * r
            disc *= gamma
            if done or trunc:
                break
        returns.append(G)
    return np.mean(returns), np.std(returns)

mean_return, std_return = run_policy(env, pi_star)
print(f"Optimal policy: mean return = {mean_return:.4f} ± {std_return:.4f}")

<div class="alert alert-warning">
Time complexity of one iteration of Value Iteration?
</div>

Note that we have repeatedly applied $\mathbb{T}^*$ to get $q^*$, and called that Value Iteration, but we could have done the same with $\mathbb{T}^\pi$ to get $q^\pi$ for a fixed $\pi$ (and there is no real canonical name for that algorithm).

# Going one step further: policy iteration and modified policy iteration.

Let's rephrase value iteration using a greediness operator.
<div class="alert alert-success">

**Greediness operator**  
For deterministic policies:
$$\pi \in \mathbb{G} q, \Leftrightarrow \pi(s) \in \arg\max_{a\in \mathcal{A}} q(s,a)$$

This can be extended to stochastic policies:
$$\pi \in \mathbb{G} q, \Leftrightarrow \pi(s) \in \arg\max_{\pi \in \Delta_\mathcal{A}} \mathbb{E}_{a\sim\pi} \left[q(s,a)\right]$$
</div>

Then, value iteration is the algorithm that defines the sequences $\pi_n$ and $q_n$ as:
<div class="alert alert-success">

**Value iteration**
$$\pi_n \in \mathbb{G} q_n, \quad q_{n+1} = \mathbb{T}^{\pi_n} q_n.$$
</div>

So we can replace $\mathbb{T}^*$ by the composition of $\mathbb{T}^\pi$ and $\mathbb{G}$. Value Iteration can be seen as picking the greedy action with respect to $q$ and then updating $q$ with just one application of $\mathbb{T}^\pi$. This application of $\mathbb{T}^\pi$ alone is not sufficient for $q$ to reach $q^\pi$ but it changes $q$ and so it changes what the greedy action will be at the next iteration.

Let's remark that acting greedily with respect to $q^\pi$ yields a better policy than $\pi$:
<div class="alert alert-success">

**Policy improvement theorem**  
If $\pi_{n+1} \in \mathbb{G}q^{\pi_n}$, then $q^{\pi_{n+1}} \geq q^{\pi_n}$.
</div>

But since $q^\pi$ is the fixed point of $\mathbb{T}^\pi$, it is the limit of the $q_{k+1} = \mathbb{T}^{\pi} q_k$ sequence.  
This provides us with an algorithm that keeps track of both a policy and a value function:

<div class="alert alert-success">

**Policy iteration**  
$$\pi_n \in \mathbb{G} q_n, \quad q_{n+1} = (\mathbb{T}^{\pi_n})^\infty q_n.$$
</div>

Obviously, an infinite number of applications of $\mathbb{T}^\pi$ is not very practical. Suppose now we only apply the $\mathbb{T}^\pi$ operator $m$ times. This provides the **Modified Policy Iteration** algorithm.
<div class="alert alert-success">

**Modified policy iteration**  
$$\pi_n \in \mathbb{G} q_n, \quad q_{n+1} = (\mathbb{T}^{\pi_n})^m q_n.$$
</div>

Interestingly, Modified Policy Iteration benefits from the same convergence properties as Policy Iteration or Value Iteration.

<div class="alert alert-warning">
What is modified policy iteration with $m=1$?
</div>

# From Exact to Approximate Value Iteration

Exact value iteration requires visiting every $(s, a)$ pair — only feasible for small discrete spaces.

For continuous or large state spaces we use **function approximation**:
$$q(s, a; w) \approx q^*(s, a).$$

<div class="alert alert-success">
    
**Approximate Value Iteration** is the algorithm that computes the sequence $q_{n+1} = \mathbb{A} \mathbb{T}^* q_n$, where $\mathbb{A}$ is an approximation procedure (turns a function into another).
</div>

Let us suppose that $\mathbb{A}$ is not a bad approximation procedure and that its approximation error is uniformly bounded, that is, 
$$\forall f \in \mathbb{R}^{\mathcal{SA}}, \ \| f-\mathbb{A}f \|_\infty \leq \epsilon.$$

The first important result is that Approximate Value Iteration **does not converge**. However, one can prove that $q_n$ reaches a neighborhood of $q^*$. Specifically, there exists $N$ such that for all $n\geq N$,
$$\| q^* - q_n \|_\infty \leq \frac{\epsilon}{1-\gamma}.$$

More importantly, let $\pi_n$ be the greedy policy with respect to $q_n$, then:
$$\|q^*-q^{\pi_n}\|_\infty \leq \frac{2\gamma}{1-\gamma} \|q^*-q_n\|_\infty.$$

And consequently, for such $n\geq N$,
$$\|q^*-q^{\pi_n}\|_\infty \leq \frac{2\gamma\epsilon}{(1-\gamma)^2}.$$

So,
<div class="alert alert-success">

Approximate Value Iteration does not necessarily converge but reaches policies whose values are close to optimal.
</div>

More on $L_\infty$ bounds for approximate DP: **[Neuro-dynamic programming](http://athenasc.com/ndpbook.html)** book by D. P. Bertsekas and J. Tsitsiklis (1996).

Most supervised learning algorithms minimize $L_n(w) = \| q_w - \mathbb{T}^* q_n \|_{2,\rho}$. There are similar convergence bounds in $L_{2,\rho}$ norm **[(Munos, 2005)](https://cdn.aaai.org/AAAI/2005/AAAI05-159.pdf)** **[(Munos, 2007)](https://epubs.siam.org/doi/abs/10.1137/040614384?journalCode=sjcodc)**.

There are also similar results for approximate modifed policy iteration **[(Scherrer et al, 2012)](https://icml.cc/2012/papers/608.pdf)**. Hence we group this general idea of approximating the sequence of value functions spanned by a Dynamic Programming procedure, under the umbrella of **Approximate Dynamic Programming**.

So far, there has been no **learning**: no data involved, no risk minimization. But the convergence properties of approximate value iteration open the door to learning the sequence $q_n$.

# Approximate value iteration as a sequence of risk minimization problems.

Recall that $q^\pi(s,a)  = \mathbb{E} [G^\pi(s,a)]$.

Let us define the **bootstrapped return** random variable:
$$G^\pi_1(s,a,q) = R_0 + \gamma q(S_1, A_1) \quad \Bigg| \quad \begin{array}{l}S_0 = s, A_0=a\\ A_1 \sim \pi(S_1),\\ S_{1}\sim p(\cdot|S_0,A_0),\\ R_0 = r(S_0,A_0,S_{1}).\end{array}$$

Then we have $(T^\pi q)(s,a) = \mathbb{E} [ G^\pi_1(s,a,q) ]$.

So VI is:
<div class="alert alert-success">

**Value iteration**
$$\pi_n \in \mathbb{G} q_n, \quad q_{n+1}(s,a) = (\mathbb{T}^{\pi_n} q_n)(s,a) = \mathbb{E} [G^\pi_1(s,a,q_n)].$$
</div>

Let's write the risk minimization problem of approximating $q_{n+1}(s,a) = \mathbb{E} [G^\pi_1(s,a,q_n)]$.

<div class="alert alert-success">

**Approximate dynamic programming as a sequence of risk minimization problems.**  
Approximate dynamic programming can be cast as finding the sequence of functions $q(s,a;w_n)$ defined by $w_{n+1} \in \arg\min_{w} L_n(w)$, with
$$L_n(w) = \frac{1}{2} \mathbb{E}_{(s,a) \sim \rho}\left[ \left( q(s,a;w) - G^\pi_1(s,a,q_n) \right)^2 \right].$$

If $L_n(w)$ differentiable,  
if one can draw iid samples $\left\{\left(s_i,a_i,r_i,s'_i\right)\right\}_{i\in [1,B]}$,   
then (SGD) $w_{n+1}$ is the limit of the sequence $w_{k+1} \leftarrow w_{k} + \alpha_k d_n(w_{k})$ with  
$$d_n(w) = \frac{1}{B} \sum_{i=1}^B \left[ \left( r_i + \gamma q(s_i',a';w_{n}) - q(s_i,a_i;w) \right) \nabla_w q(s_i,a_i;w) \right].$$
</div>

**Sources of error in the approximation procedure**  
Note that three independent factors might prevent us from actually learning $q^\pi$:
1. $q^\pi$ might not live in the set of parametric functions $q_w$.
2. The SGD updates might converge to a non-zero empirical risk (and to a non-zero risk).
3. $\rho$ might not cover appropriately the whole span of $\mathcal{S}\times \mathcal{A}$.

Factor 1 begs for good approximation methods, with little bias: universal function approximators will play a big role in value function learning.  
Factor 2 begs for good SGD optimizers and maybe for regularization of $L_n(w)$.  
Factor 3 warns us about the interplay between collected samples and minimization of $L_n$ (which has consequences both for online and offline RL). 

Note that minimizing the empirical risk does not require it to be differentiable with respect to the parameters of $q$. For instance, one could use decision trees or **[random forests](https://link.springer.com/article/10.1023/A:1010933404324)** for this purpose.  
Note also that other objective functions can be used instead of the empirical risk, like regularized risk measures (as in **[support vector regression](https://link.springer.com/article/10.1023/B:STCO.0000035301.49549.88)** for instance).

The goal of this section was to state an important idea: 
<div class="alert alert-success">

Approximate dynamic programming can be tackled as a sequence of supervised learning problems.
</div>

And in particular:

<div class="alert alert-success">

**Approximate value iteration as a sequence of risk minimization problems**  
$$\pi_n \in \mathbb{G} q_n,$$
$$L_n(w) = \frac{1}{2} \mathbb{E}_{(s,a) \sim \rho}\left[ \left( q(s,a;w) - G^{\pi_n}_1(s,a,q_n) \right)^2 \right],$$
$$w_{n+1} \in \arg\min_{w} L_n(w),$$
$$q_{n+1}(s,a) = q(s,a;w_{n+1}).$$
</div>

<div class="alert alert-warning">
What happens if we take a fixed $\pi$ and remove the $\pi_n \in \mathbb{G} q_n$ step in the procedure above?
</div>

Variations on the above procedure:
- Remove the $\pi_n \in \mathbb{G} q_n$ step and take a fixed $\pi$ $\rightarrow$ approximate the $q_{n+1}=\mathbb{T}^\pi q_n$ sequence
- Perform gradient descent on $L_n$ with a fixed $\pi$, one sample at a time (stochastic approximation) $\rightarrow$ **temporal difference** (TD(0)) algorithms
- Put back the $\pi_n \in \mathbb{G} q_n$ step and perform stochastic approximation $\rightarrow$ **Q-learning**
- Store a buffer of samples to perform SGD on $L_n$ $\rightarrow$ **Deep Q-Networks** (DQN)

<div class="alert alert-warning">
When we perform SGD to minimize $L_n$, we need samples across $\mathcal{S}\times\mathcal{A}$. How do we make sure we get good samples?
</div>

<div class="alert alert-warning">
Caveat: can you spot the problem with the iid samples assumption (of SGD), when the samples are obtained from interaction with the system to control?
</div>

# Deep Q-Networks (DQN)

DQN **[(Mnih et al, 2013)](https://arxiv.org/abs/1312.5602)** **[(Mnih et al, 2015)](https://deepmind.com/research/publications/human-level-control-through-deep-reinforcement-learning)** is AVI with neural network function approximation and two key tricks.

## Experience Replay

Store transitions $(s, a, r, s', d)$ in a **replay buffer** [**(Lin, 1992)**](https://link.springer.com/article/10.1007/BF00992699) and sample **random mini-batches** to break temporal correlations.

## Target Network

Keep a **frozen copy** $\hat{q}(\cdot; \theta^-)$ of the network for computing targets.  
Update $\theta^-$ periodically (copy $\theta$ every $C$ steps).  
This stabilises training by making targets move more slowly.  

But, really, this is just a renaming of $q_n$ and this is vanilla AVI.

## $\epsilon$-greedy exploration

Take a random action with probability $\epsilon$ (decreasing over training), greedy action otherwise.
This ensures the replay buffer contains diverse transitions.

## Implementation

In [ ]:
import gymnasium as gym
env_cp = gym.make('CartPole-v1', render_mode="rgb_array")
print("CartPole state space :", env_cp.observation_space)
print("CartPole action space:", env_cp.action_space)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

env_cp.reset()
plt.imshow(env_cp.render());

In [ ]:
import random
import torch
import numpy as np

class ReplayBuffer:
    """Fixed-size FIFO replay buffer returning torch Tensors."""
    def __init__(self, capacity, device):
        self.capacity = int(capacity)
        self.data = []
        self.index = 0
        self.device = device

    def append(self, s, a, r, s_next, done):
        if len(self.data) < self.capacity:
            self.data.append(None)
        self.data[self.index] = (s, a, r, s_next, done)
        self.index = (self.index + 1) % self.capacity

    def sample(self, batch_size):
        batch = random.sample(self.data, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (
            torch.FloatTensor(np.array(s)).to(self.device),
            torch.LongTensor(np.array(a)).to(self.device),
            torch.FloatTensor(np.array(r)).to(self.device),
            torch.FloatTensor(np.array(s2)).to(self.device),
            torch.FloatTensor(np.array(d)).to(self.device),
        )

    def __len__(self):
        return len(self.data)

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def make_dqn(state_dim, n_actions, nb_neurons=128):
    return nn.Sequential(
        nn.Linear(state_dim, nb_neurons), nn.ReLU(),
        nn.Linear(nb_neurons, nb_neurons), nn.ReLU(),
        nn.Linear(nb_neurons, n_actions)
    ).to(device)

state_dim = env_cp.observation_space.shape[0]
n_actions = env_cp.action_space.n
model = make_dqn(state_dim, n_actions)
print(model)

In [ ]:
import copy
from tqdm import trange

def greedy_action(network, state):
    with torch.no_grad():
        s = torch.FloatTensor(state).unsqueeze(0).to(device)
        return network(s).argmax().item()

class DQNAgent:
    def __init__(self, env, config):
        state_dim = env.observation_space.shape[0]
        n_actions = env.action_space.n
        self.n_actions   = n_actions
        self.gamma       = config.get('gamma', 0.99)
        self.batch_size  = config.get('batch_size', 64)
        self.eps_max     = config.get('eps_max', 1.0)
        self.eps_min     = config.get('eps_min', 0.05)
        self.eps_delay   = config.get('eps_delay', 100)
        self.eps_period  = config.get('eps_period', 5000)
        self.target_freq = config.get('target_update_freq', 200)
        self.epsilon     = self.eps_max
        self.eps_step    = (self.eps_max - self.eps_min) / self.eps_period

        self.model  = make_dqn(state_dim, n_actions)
        self.target = copy.deepcopy(self.model)
        self.memory = ReplayBuffer(config.get('buffer_size', 50_000), device)
        self.optim  = torch.optim.Adam(self.model.parameters(),
                                       lr=config.get('lr', 1e-3))
        self.loss_fn = nn.SmoothL1Loss()
        self.steps = 0

    def act(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return greedy_action(self.model, state)

    def learn_step(self):
        if len(self.memory) < self.batch_size:
            return
        s, a, r, s2, done = self.memory.sample(self.batch_size)
        with torch.no_grad():
            target_vals = r + self.gamma * (1 - done) * self.target(s2).max(1).values
        current_vals = self.model(s).gather(1, a.unsqueeze(1)).squeeze(1)
        loss = self.loss_fn(current_vals, target_vals)
        self.optim.zero_grad()
        loss.backward()
        self.optim.step()

    def train(self, env, n_episodes):
        episode_returns = []
        for ep in trange(n_episodes):
            state, _ = env.reset()
            ep_return = 0.0
            done = False
            while not done:
                action = self.act(state)
                next_state, reward, done, trunc, _ = env.step(action)
                self.memory.append(state, action, reward, next_state, float(done or trunc))
                self.learn_step()
                state = next_state
                ep_return += reward
                self.steps += 1
                # Epsilon decay
                if self.steps > self.eps_delay:
                    self.epsilon = max(self.eps_min, self.epsilon - self.eps_step)
                # Target network update
                if self.steps % self.target_freq == 0:
                    self.target.load_state_dict(self.model.state_dict())
                if trunc:
                    break
            episode_returns.append(ep_return)
        return episode_returns

In [ ]:
config = {
    'gamma': 0.99,
    'lr': 1e-3,
    'batch_size': 64,
    'buffer_size': 50_000,
    'eps_max': 1.0,
    'eps_min': 0.05,
    'eps_delay': 50,
    'eps_period': 3000,
    'target_update_freq': 200,
}

agent = DQNAgent(env_cp, config)
returns = agent.train(env_cp, n_episodes=300)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

def smooth(x, w=20):
    return np.convolve(x, np.ones(w)/w, mode='valid')

plt.figure(figsize=(9, 3))
plt.plot(returns, alpha=0.3, label='episode return')
plt.plot(smooth(returns), label=f'{20}-ep moving avg')
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title("DQN on CartPole-v1")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Final 50-episode average: {np.mean(returns[-50:]):.1f}")

In [ ]:
# Watch the trained agent
from gymnasium.utils.save_video import save_video
import os

test_env = gym.make("CartPole-v1", render_mode="rgb_array_list")
state, _ = test_env.reset()
for _ in range(500):
    action = greedy_action(agent.model, state)
    state, _, done, trunc, _ = test_env.step(action)
    if done or trunc:
        break

os.makedirs("videos", exist_ok=True)
save_video(test_env.render(), "videos",
           fps=test_env.metadata["render_fps"],
           name_prefix="dqn_cartpole")
test_env.close()

In [ ]:
from IPython.display import Video
Video("videos/dqn_cartpole-episode-0.mp4")

## DQN discussion

DQN works well when:
- The action space is **discrete and small** (the $\max_a$ operation is cheap).
- The state can be represented as a fixed-size vector (or image).

**What if the action space is continuous?**  
FrozenLake has 4 actions; CartPole has 2.  
The V2G problem has a **10-dimensional continuous action space** — taking the max over $a$ is no longer trivial.  
→ This is where **actor-critic methods** come in.

## Extensions of DQN

- mitigating overestimation effects ([double Q-functions](https://ojs.aaai.org/index.php/AAAI/article/view/10295))
- n-step returns
- distributional RL (eg. [C51](http://proceedings.mlr.press/v70/bellemare17a.html?trk=public_post_comment-text))
- better exploration (eg. [NGU](https://arxiv.org/abs/2002.06038))
- importance sampling in the replay buffer ([PER](https://arxiv.org/abs/1511.05952) or [LaBER](https://proceedings.mlr.press/v162/lahire22a/lahire22a.pdf))  
- regularization and action-gap spreading ([Munchausen DQN](https://proceedings.neurips.cc/paper/2020/hash/2c6a0bae0f071cbbf0bb3d5b11d90a82-Abstract.html))

And lots more.  
Lesson learned: things are well engineered but there are still things to explore.

# Discussion: the key challenges of RL

Playing with DQN has highlighted the three fundamental challenges in RL. Let's discuss them.

<div class="alert alert-success">

**Intrinsic challenges in RL:**  
- function approximation,
- the improvement problem,
- the exploration versus exploitation trade-off.
</div>

# Opening towards continuous actions in AVI

## Monotonicity of the policy sequence

For **continuous action spaces**, we cannot enumerate $\max_a q^*(s, a)$.

**Actor-critic** methods maintain two networks:
- **Critic** $q(s, a)$: approximates the Q-function.
- **Actor** $\pi(a|s)$: outputs an action (or a distribution) directly.

Overall goal: preserve the monotonicity of the policy improvement theorem, i.e. the sequence of $\pi$ should be monotonous in $J(\pi)$.

<div class="alert alert-success">

**[Deterministic Policy Gradient](https://proceedings.mlr.press/v32/silver14.html)** theorem (Silver et al, 2014)  
Consider a deterministic policy $\pi_\theta: \mathcal{S}\rightarrow \mathcal{A}$ interacting with an MDP $(\mathcal{S}, \mathcal{A}, p, r)$ with a starting state distribution $\rho_0$.  
We will drop the $\theta$ subscripts wherever unambiguous, to improve readability.    
If $p(s,a)$, $\nabla_a p(s'|s,a)$, $r(s,a)$, $\nabla_a r(s,a)$, $\rho_0(s)$, $\pi_w(s)$, and $\nabla_\theta\pi_\theta(s)$ all exist and are continuous in $(s,a,s')$, then 
$$\nabla_\theta J(\theta) = \mathbb{E}_{s\sim \rho^{\pi}} \left[ \nabla_a q^{\pi}(s,a)|_{a=\pi(s)} \cdot \nabla_\theta \pi_\theta(s) \right].$$
</div>
    
Quite restrictive hypothesis for the DPG theorem to be applicable.  
Actually, limit case of the stochastic PG theorem (next notebook).

<div class="alert alert-success">

**[Deep Deterministic Policy Gradient](https://arxiv.org/abs/1509.02971)** (Lillicrap et al, 2016)  
With minibatch Monte Carlo gradient estimates:
$$\begin{array}{rl}
\textrm{Policy improvement} & \left\{\begin{array}{l} 
L^\pi(\theta) = \mathbb{E}_{s\sim\rho} \left[ q_w(s,\pi_\theta(s)) \right]\\ 
\theta \leftarrow \theta +\alpha \nabla_\theta L^\pi(\theta)\\ 
\end{array}\right.\\
\textrm{Q-function estimation} & \left\{\begin{array}{l} 
L^q(w) = \frac{1}{2} \mathbb{E}_{(s,a) \sim \rho}\left[ \left( q_w(s,a) - G^{\pi_{\theta'}}_1(s,a,q_{w'}) \right)^2 \right]\\
w \leftarrow w - \alpha \nabla_w L^q(w)\\
\end{array}\right.
\end{array}$$
And the target networks are updated according to:
$$\theta' \leftarrow \tau \theta + (1-\tau) \theta',$$
$$w' \leftarrow \tau w + (1-\tau) w'.$$
</div>

Going further: [Twin Delayed DDPG](https://arxiv.org/pdf/1802.09477.pdf) (TD3, Fujimoto et al, 2018).  

So, in short, the actor maximises the Q-function *implicitly* through gradient ascent on $L^\pi$.

<div class="alert alert-warning">
Tricky question: what is the distribution on $s$ in the policy update? Why is it a sensitive topic?
</div>

## Soft-actor critic: entropy-regularized RL with continuous actions

Why stochastic policies?
- "smoother" optimization landscape
- stoch $\pi$ is a form of memory of candidate optimal actions

$\rightarrow$ keep some amount of entropy in the policy

This is called **entropy-regularized RL** or **maximum entropy (maxEnt) RL**:
$$J(\pi) = \mathbb{E}\left[ \sum_t \gamma^t \left( r(s_t, a_t) + \alpha\, \mathcal{H}(\pi(\cdot|s_t)) \right) \right]$$
where $\mathcal{H}(\pi(\cdot|s)) = -\mathbb{E}_{a\sim\pi}[\log \pi(a|s)]$ is the policy **entropy** in $s$ and $\alpha$ is a temperature parameter.

<div class="alert alert-success">

**[Soft Policy Iteration](https://arxiv.org/abs/1812.05905)** (Haarnoja et al, 2019)

Entropy-regularized Bellman operator:
$$\mathbb{T}^\pi q(s,a) = r(s,a) + \gamma \mathbb{E}_{s'\sim p(\cdot|s,a)} \mathbb{E}_{a'\sim \pi(s')} \left[ q(s',a') - \alpha \log \pi(a'|s') \right].$$

Monotonic policy improvement:
$$\pi_{n+1}(s) = \arg\min_{\pi \in \Pi} D_{KL} \left( \pi(s) \Bigg|\Bigg| \frac{\exp(\frac{1}{\alpha}q^{\pi_n}(s,a))}{Z^{\pi_n}(s)} \right).$$
</div>

<div class="alert alert-success">

**[Soft Actor Critic (SAC)](https://arxiv.org/abs/1812.05905)** (Haarnoja et al, 2019)  

$$L^q(w) = \frac{1}{2} \mathbb{E}_{(s,a)\sim \rho} \left[ \left( r(s,a) + \gamma \mathbb{E}_{\substack{s'\sim p(\cdot|s,a)\\a'\sim\pi_{\theta}(s')}}\left[ q(s',a';w') - \alpha \log \pi_{\theta}(a'|s') \right] - q(s,a;w) \right)^2 \right]$$

$$L^\pi(\theta) = \mathbb{E}_{\substack{s\sim \rho\\a\sim\pi_\theta(s)}} \left[ \alpha \log \pi_\theta(a|s) - q(s,a;w) \right] $$
Alternate gradient steps on each.
</div>

<div class="alert alert-warning">
Tricky question: is SAC rather API or AVI?
</div>

Alternate formulation: minimum amount of entropy in each state $\forall s, \mathcal{H}(\pi(s)) \geq H$.

Caveat: SAC is often described as "add an entropy bonus to encourage exploration" but that's a quite inaccurate shortcut morally. It's more of a regularization term for the maximization problem, than an exploration bonus term (which is more of a nice side effect).

Key SAC ingredients in practice:
- **Squashed Gaussian actor**: $a = \tanh(\mu + \sigma \epsilon)$, $\epsilon \sim \mathcal{N}(0,I)$.
- **Two critics** (to reduce overestimation bias).
- **Automatic entropy tuning**: $\alpha$ is adjusted so that $\mathcal{H}(\pi) \approx$ target entropy.

In practice, SAC is the go-to *off-policy* algorithm for continuous control.

# Back to V2G

<div class="alert alert-warning">

**Discussion and mini-exercise**
1. Can we apply tabular value iteration to V2G? Why or why not?
2. Can we apply DQN directly to V2G? What modification would be needed?
3. Which algorithm from this notebook seems most promising for V2G? Why?
</div>

In [4]:
# Let's benchmark random vs ChargeAsFastAsPossible on V2G
from ev2gym.models.ev2gym_env import EV2Gym
from ev2gym.rl_agent.state import V2G_profit_max
from ev2gym.baselines.heuristics import ChargeAsFastAsPossible
import numpy as np

env_v2g = EV2Gym(config_file="custom.yaml", save_replay=False,
                 save_plots=False, state_function=V2G_profit_max)

def eval_v2g(agent_fn, n_episodes=20):
    """Evaluate an agent on V2G; agent_fn(env) -> action array."""
    returns = []
    for _ in range(n_episodes):
        state, _ = env_v2g.reset()
        G = 0.0
        for _ in range(env_v2g.simulation_length):
            action = agent_fn(env_v2g)
            state, r, done, trunc, _ = env_v2g.step(action)
            G += r
            if done or trunc:
                break
        returns.append(G)
    return np.mean(returns), np.std(returns)

cafap = ChargeAsFastAsPossible()
mean_h, std_h = eval_v2g(lambda e: cafap.get_action(e))
mean_r, std_r = eval_v2g(lambda e: e.action_space.sample())

print(f"Heuristic (CAFAP): {mean_h:.2f} ± {std_h:.2f}")
print(f"Random policy    : {mean_r:.2f} ± {std_r:.2f}")
print()
print("An RL agent trained with SAC should outperform both.")
print("SAC is available via stable-baselines3:")
print("  from stable_baselines3 import SAC")
print("  model = SAC('MlpPolicy', env_v2g, verbose=1)")
print("  model.learn(total_timesteps=100_000)")

Heuristic (CAFAP): -10083.43 ± 6786.55
Random policy    : -12624.99 ± 3435.72

An RL agent trained with SAC should outperform both.
SAC is available via stable-baselines3:
  from stable_baselines3 import SAC
  model = SAC('MlpPolicy', env_v2g, verbose=1)
  model.learn(total_timesteps=100_000)


In [2]:
%pip install 'stable-baselines3[extra]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 466.9 kB/s  0:00:11 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 401.9 kB/s  0:00:16 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 2.2 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [stable-baselines3]stable-baselines3]
Note: you may need to restart the kernel to use updated packages.


In [5]:
from ev2gym.models.ev2gym_env import EV2Gym
from ev2gym.rl_agent.state import V2G_profit_max
import numpy as np

env_v2g = EV2Gym(config_file="custom.yaml", save_replay=False,
                 save_plots=False, state_function=V2G_profit_max)

In [6]:
from stable_baselines3 import SAC
model = SAC('MlpPolicy', env_v2g, verbose=1)
model.learn(total_timesteps=100_000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 80        |
|    ep_rew_mean     | -1.04e+04 |
| time/              |           |
|    episodes        | 4         |
|    fps             | 72        |
|    time_elapsed    | 4         |
|    total_timesteps | 320       |
| train/             |           |
|    actor_loss      | 174       |
|    critic_loss     | 6.14e+04  |
|    ent_coef        | 0.939     |
|    ent_coef_loss   | -0.716    |
|    learning_rate   | 0.0003    |
|    n_updates       | 219       |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 80        |
|    ep_rew_mean     | -7.87e+03 |
| time/              |           |
|    episodes        | 8         |
|    fps             | 61        |
|    time_elapsed    | 10        |
|    total_timesteps | 640     

In [ ]:
model.save("sac_v2g_model")

In [ ]:
#model = SAC.load("sac_v2g_model", env=env_v2g)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Arrays to store evaluation data
episode_returns = []

# Run 10 evaluation rollouts
num_episodes = 10

for episode in range(num_episodes):
    # Reset the environment at the start of each episode
    obs, info = env_v2g.reset()
    done = False
    episode_reward = 0
    
    while not done:
        # Predict the action using the loaded policy
        # deterministic=True is standard for evaluation/deployment
        action, _states = model.predict(obs, deterministic=True)
        
        # Step through the environment
        obs, reward, terminated, truncated, info = env_v2g.step(action)
        
        episode_reward += reward
        done = terminated or truncated
        
    episode_returns.append(episode_reward)
    print(f"Episode {episode + 1}: Total Return = {episode_reward:.2f}")

# --- Plotting the Results ---
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_episodes + 1), episode_returns, marker='o', linestyle='-', color='b')
plt.title('V2G Policy Performance Over 10 Rollouts')
plt.xlabel('Episode')
plt.ylabel('Total Return')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(range(1, num_episodes + 1))
plt.show()

# Summary

- Dynamic programming (Bellman equations, VI, PI, MPI)
- Approximate dynamic programming (non-convergence, bounds)
- ADP as a sequence of risk minimization problems (TD(0), Q-learning, DQN)
- Continuous actions in AVI, regularization (DDPG, TD3, SAC)

| Method | State space | Action space | Key idea |
|---|---|---|---|
| Value Iteration | Discrete, tabular | Discrete | Exact Bellman contraction |
| DQN | Continuous | Discrete | Neural Q + replay + target net |
| SAC | Continuous | Continuous | Actor-critic + entropy reg. |

In the next notebook we will take a fundamentally different approach: instead of learning the value function and extracting a policy, we will **directly optimise the policy** using **policy gradient** methods.